# MoRoOp Dataset Walkthrough

This notebook demonstrates the public API of `moroop-dataset-toolkit`. It loads the local MoRoOp Parquet files and recreates the four publication figures without downloading data by default.

Install the package first with `python -m pip install -e .` from the repository root.

In [ ]:
from pathlib import Path

from moroop_dataset_toolkit import (
    DEFAULT_HF_REPOSITORY,
    download_from_huggingface,
    download_from_kth_repository,
    load_table,
)
from moroop_dataset_toolkit.figures import (
    battery_state,
    driving_path,
    representative_gantt,
    representative_velocity,
)

DATA_DIRECTORY = Path("dataset")
SHIFT = "2026_07_27_evening"
RELEASE_TAG = "1.0.1"
FIGURES_DIRECTORY = Path("figures")

## Download a versioned release

The following calls are intentionally commented out. Uncomment exactly one after choosing the distribution source. Hugging Face supports immutable release tags; the KTH download requires the direct ZIP URL from the published repository record.

In [ ]:
destination = Path("dataset")
download_from_huggingface(
    destination,
    repository=DEFAULT_HF_REPOSITORY,
    revision=RELEASE_TAG,
)

# version 1.0.1 of the dataset is also available through the KTH Data Repository
download_from_kth_repository(
    destination,
    archive_url="https://datarepository.kth.se/records/vx0cq-w7h93/files/MoRoOp_V_1_0_1.zip"
)

DATA_DIRECTORY = destination / "data"

## Load and inspect data

Load shift-level records and the shared kit catalogue through `load_table()`.

In [ ]:
jobs = load_table(DATA_DIRECTORY, "jobs", shift=SHIFT)
operations = load_table(DATA_DIRECTORY, "operations", shift=SHIFT)
robot_states = load_table(
    DATA_DIRECTORY,
    "robot_state_cleaned",
    shift=SHIFT,
    columns=["id", "created_at", "pos_x", "pos_y", "state"],
)
kits = load_table(DATA_DIRECTORY, "kits")

print(f"Hugging Face repository: {DEFAULT_HF_REPOSITORY}")
print(f"Shift: {SHIFT}")
print(f"jobs: {len(jobs):,} rows")
print(f"operations: {len(operations):,} rows")
print(f"cleaned robot states: {len(robot_states):,} rows")
print(f"kit components: {len(kits):,} rows")

jobs.head()

In [ ]:
assert list(kits.columns) == ["kit_id", "color", "size", "quantity"]
assert jobs["created_at"].is_monotonic_increasing
assert operations["created_at"].is_monotonic_increasing

kits.groupby("kit_id")["color"].nunique().rename("distinct_colors").head()

## Recreate publication figures

The package functions write four PDF figures using the representative data and parameters from the publication.

In [ ]:
FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)

representative_gantt(
    DATA_DIRECTORY,
    FIGURES_DIRECTORY / "representative_operations_gantt.pdf",
)
representative_velocity(
    DATA_DIRECTORY,
    FIGURES_DIRECTORY / "representative_operations_velocity.pdf",
)
battery_state(DATA_DIRECTORY, FIGURES_DIRECTORY / "robot_battery_state.pdf")
driving_path(DATA_DIRECTORY, FIGURES_DIRECTORY / "robot_driving_path.pdf")

In [ ]:
expected_figures = [
    "representative_operations_gantt.pdf",
    "representative_operations_velocity.pdf",
    "robot_battery_state.pdf",
    "robot_driving_path.pdf",
]

for filename in expected_figures:
    path = FIGURES_DIRECTORY / filename
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty output: {path}"
    print(f"{filename}: {path.stat().st_size:,} bytes")

## Validation errors

The loader rejects invalid table names and enforces that shift-level tables are requested with a shift identifier.

In [ ]:
for table, shift in [("not_a_table", SHIFT), ("jobs", None), ("kits", SHIFT)]:
    try:
        load_table(DATA_DIRECTORY, table, shift=shift)
    except ValueError as error:
        print(f"{table!r}, shift={shift!r}: {error}")
    else:
        raise AssertionError("Expected load_table to reject the invalid request.")